# 02. Скачивание и подготовка ключевой ставки Банка России

Ноутбук скачивает значения ключевой ставки Банка России через SOAP-сервис ЦБ РФ, приводит данные к единому формату и сохраняет файл для дальнейшего объединения с дневными данными ОФЗ.

Выходные файлы:

- `data/raw/key_rate/cbr_key_rate_2016_2026.csv` — исходная таблица, полученная с сайта Банка России;
- `data/processed/key_rate_2016_2026.csv` — подготовленная таблица для объединения с данными ОФЗ.


## 1. Импорт библиотек и пути проекта

In [1]:
from pathlib import Path
import xml.etree.ElementTree as ET

import requests
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

In [2]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
SUMMARY_DIR = DATA_DIR / "summary"
KEY_RATE_RAW_DIR = RAW_DIR / "key_rate"

for path in [RAW_DIR, PROCESSED_DIR, SUMMARY_DIR, KEY_RATE_RAW_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT.resolve())
print("Raw key-rate dir:", KEY_RATE_RAW_DIR.resolve())
print("Processed data dir:", PROCESSED_DIR.resolve())

Project root: /Users/daniilsestov/Desktop/курсач для гита/ofz_git_notebooks_daily_only
Raw key-rate dir: /Users/daniilsestov/Desktop/курсач для гита/ofz_git_notebooks_daily_only/data/raw/key_rate
Processed data dir: /Users/daniilsestov/Desktop/курсач для гита/ofz_git_notebooks_daily_only/data/processed


## 2. Параметры периода

In [3]:
FROM_DATE = "2016-05-03"
TO_DATE = "2026-05-03"

RAW_KEY_RATE_PATH = KEY_RATE_RAW_DIR / "cbr_key_rate_2016_2026.csv"
PROCESSED_KEY_RATE_PATH = PROCESSED_DIR / "key_rate_2016_2026.csv"

print("Period:", FROM_DATE, "—", TO_DATE)

Period: 2016-05-03 — 2026-05-03


## 3. Запрос к SOAP-сервису Банка России

In [4]:
CBR_URL = "https://www.cbr.ru/DailyInfoWebServ/DailyInfo.asmx"

soap_body = f'''<?xml version="1.0" encoding="utf-8"?>
<soap:Envelope
    xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
    xmlns:xsd="http://www.w3.org/2001/XMLSchema"
    xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/">
  <soap:Body>
    <KeyRateXML xmlns="http://web.cbr.ru/">
      <fromDate>{FROM_DATE}T00:00:00</fromDate>
      <ToDate>{TO_DATE}T00:00:00</ToDate>
    </KeyRateXML>
  </soap:Body>
</soap:Envelope>
'''

headers = {
    "Content-Type": "text/xml; charset=utf-8",
    "SOAPAction": "http://web.cbr.ru/KeyRateXML",
}

response = requests.post(CBR_URL, data=soap_body.encode("utf-8"), headers=headers, timeout=60)
response.raise_for_status()

print("HTTP status:", response.status_code)
print("Response size, bytes:", len(response.content))

HTTP status: 200
Response size, bytes: 152202


## 4. Разбор XML и формирование таблицы

In [5]:
root = ET.fromstring(response.content)


def clean_tag(tag: str) -> str:
    return tag.split("}")[-1]


rows = []

for elem in root.iter():
    children = list(elem)
    child_names = {clean_tag(child.tag) for child in children}

    if {"DT", "Rate"}.issubset(child_names):
        row = {}
        for child in children:
            row[clean_tag(child.tag)] = child.text
        rows.append(row)

key_rate = pd.DataFrame(rows)

if key_rate.empty:
    raise RuntimeError("Сервис ЦБ не вернул строки с полями DT и Rate. Нужно проверить ответ response.content.")

key_rate = key_rate.rename(columns={
    "DT": "date",
    "Rate": "key_rate",
})

key_rate["date"] = pd.to_datetime(key_rate["date"], errors="coerce")
key_rate["key_rate"] = pd.to_numeric(key_rate["key_rate"], errors="coerce")

key_rate = (
    key_rate[["date", "key_rate"]]
    .dropna(subset=["date", "key_rate"])
    .drop_duplicates(subset=["date"])
    .sort_values("date")
    .reset_index(drop=True)
)

key_rate = key_rate[(key_rate["date"] >= FROM_DATE) & (key_rate["date"] <= TO_DATE)].copy()

print("Prepared shape:", key_rate.shape)
display(key_rate.head())
display(key_rate.tail())

Prepared shape: (2514, 2)


,date,key_rate
0,2016-05-04 00:00:00+03:00,11.0
1,2016-05-05 00:00:00+03:00,11.0
2,2016-05-06 00:00:00+03:00,11.0
3,2016-05-10 00:00:00+03:00,11.0
4,2016-05-11 00:00:00+03:00,11.0


,date,key_rate
2509,2026-04-24 00:00:00+03:00,15.0
2510,2026-04-27 00:00:00+03:00,14.5
2511,2026-04-28 00:00:00+03:00,14.5
2512,2026-04-29 00:00:00+03:00,14.5
2513,2026-04-30 00:00:00+03:00,14.5


## 5. Сохранение файлов

In [6]:
# Сохраняем и сырой результат, и подготовленный файл. По содержанию они почти совпадают,
# но разнесены по папкам raw/processed для аккуратной структуры Git.
key_rate.to_csv(RAW_KEY_RATE_PATH, index=False, encoding="utf-8-sig")
key_rate.to_csv(PROCESSED_KEY_RATE_PATH, index=False, encoding="utf-8-sig")

print("Saved raw key rate:", RAW_KEY_RATE_PATH.resolve())
print("Saved processed key rate:", PROCESSED_KEY_RATE_PATH.resolve())
print("Date range:", key_rate["date"].min(), "—", key_rate["date"].max())

Saved raw key rate: /Users/daniilsestov/Desktop/курсач для гита/ofz_git_notebooks_daily_only/data/raw/key_rate/cbr_key_rate_2016_2026.csv
Saved processed key rate: /Users/daniilsestov/Desktop/курсач для гита/ofz_git_notebooks_daily_only/data/processed/key_rate_2016_2026.csv
Date range: 2016-05-04 00:00:00+03:00 — 2026-04-30 00:00:00+03:00
